# Viterbi semantic labelling of a plant point cloud

*Converted from `script/Viterbi_Workflow.py`.*

*Archived source: `script/Viterbi_Workflow.py` @ commit `428f576` — restore with `git show 428f576:script/Viterbi_Workflow.py`.*


This notebook labels each quotient-graph node of a segmented plant cloud as **leaf / stem / NSP**
(no-simple-part) using a **Hidden Markov Model decoded with the Viterbi algorithm**, run over the
spanning tree of the quotient graph. The observations are geometric descriptors (planarity, linearity,
scattering) computed from the local point-cloud covariance at each node.


## 1. Imports

In [ ]:
import numpy as np
import math
import time
import os
from collections import Counter, deque

import open3d as o3d
import spectral_clustering.similarity_graph as sgk
import networkx as nx

from spectral_clustering.point_cloud_graph import *
from spectral_clustering.quotient_graph import *
from spectral_clustering.topological_energy import *
from spectral_clustering.segmentation_algorithms import *
from spectral_clustering.split_and_merge import *
from spectral_clustering.display_and_export import *
from spectral_clustering.quotientgraph_semantics import *
from spectral_clustering import Viterbi as V


In [ ]:
# All outputs (figures, .txt exports) are written to the local `outputs/`
# sub-folder, which is git-ignored (see `.gitignore`). Library export
# functions receive `OUTPUT_DIR` explicitly instead of relying on a global
# `os.chdir`.
from pathlib import Path
OUTPUT_DIR = Path.cwd() / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

## 2. Segment the cloud (geometry-only, minimal)

Build the connected Riemannian graph and run the coarse spectral segmentation to get the point-cloud
`clustering_labels` needed to build the quotient graph. This mirrors the pipeline notebook.

`@time`: kNN graph ~10-30s, eigen/gradient/kmeans ~15s.


In [ ]:
# @time (~40s)
begin = time.time()
model_filename = '../data/chenos/cheno_3.ply'
filenamemod = os.path.splitext(os.path.basename(model_filename))[0]

In [ ]:
pcd = o3d.io.read_point_cloud(model_filename, format='ply')
r = 18
SimG, pcdfinal = sgk.create_connected_riemannian_graph(point_cloud=pcd, method='knn', nearest_neighbors=r)
G = PointCloudGraph(SimG)
G.pcd = pcdfinal

In [ ]:
def detect_lower_point(G):
    a = np.amin(G.nodes_coords[:, 2], axis=0)
    result = np.where(G.nodes_coords[:, 2] == a)
    return result[0][0]
root_point_riemanian = detect_lower_point(G)

In [ ]:
G.compute_graph_eigenvectors()
G.compute_gradient_of_fiedler_vector(method='by_fiedler_weight')
clusters_centers = G.clustering_by_kmeans_using_gradient_norm(export_in_labeled_point_cloud=True, number_of_clusters=4, directory=OUTPUT_DIR)
label_leaves = select_minimum_centroid_class(clusters_centers)
print('points:', len(G.nodes), 'root:', root_point_riemanian)

## 3. Merge small clusters

`compute_local_descriptors` estimates a covariance per QG node and fails on degenerate (<4 points)
segments. Merge them into the nearest neighbouring cluster through point-graph adjacency so every
node has enough points.


In [ ]:
def merge_small_clusters(G, labels, min_size=4):
    """Reassign points of clusters smaller than min_size to the nearest
    neighbouring cluster via point-cloud graph adjacency."""
    labels = np.asarray(labels).ravel().astype(int)
    sizes = {c: int((labels == c).sum()) for c in np.unique(labels)}
    src = np.full(len(labels), -1, dtype=int)
    q = deque()
    for i in range(len(labels)):
        if sizes[labels[i]] >= min_size:
            q.append(i); src[i] = labels[i]
    while q:
        u = q.popleft()
        for v in G.neighbors(u):
            if src[v] == -1:
                src[v] = src[u]; q.append(v)
    src[src == -1] = labels[src == -1]
    return src.reshape(-1, 1)

In [ ]:
merged_labels = merge_small_clusters(G, G.kmeans_labels_gradient, min_size=4)
QG = QuotientGraph()
QG.build_from_pointcloudgraph(G, merged_labels)
QG.compute_nodes_coordinates()

In [ ]:
def merge_small_qg_nodes(G, QG, min_size=4):
    """Reassign points of QG nodes smaller than min_size to the nearest larger
    neighbouring QG node via point-cloud graph adjacency, then rebuild the QG.

    build_from_pointcloudgraph region-grows each cluster label into connected
    components, so a single large cluster can fragment into tiny (<min_size) QG
    nodes. These must be merged or covariance descriptors below will fail."""
    order = list(G.nodes())
    memberships = np.asarray([G.nodes[u]['quotient_graph_node'] for u in order], dtype=int).ravel()
    sizes = {n: int(QG.nodes[n]['intra_class_node_number']) for n in QG.nodes}
    idx = {u: i for i, u in enumerate(order)}
    src = np.full(len(order), -1, dtype=int)
    q = deque()
    for i, u in enumerate(order):
        if sizes[memberships[i]] >= min_size:
            src[i] = memberships[i]; q.append(i)
    while q:
        i = q.popleft()
        u = order[i]
        for v in G.neighbors(u):
            j = idx[v]
            if src[j] == -1:
                src[j] = src[i]; q.append(j)
    src[src == -1] = memberships[src == -1]
    for i, u in enumerate(order):
        G.nodes[u]['quotient_graph_node'] = int(src[i])
    QG.rebuild(G)
    sz = {n: QG.nodes[n]['intra_class_node_number'] for n in QG.nodes}
    return len(QG.nodes), sum(1 for v in sz.values() if v < min_size)

In [ ]:
n, small = merge_small_qg_nodes(G, QG, min_size=4)
print('QG nodes:', n, '| nodes <4 pts:', small)

## 4. Local geometric descriptors

Compute planarity / linearity / scattering per QG node from the local covariance (neighbourhood =
`pointcloudgraph`, scale 10). These are the HMM observations.


In [ ]:
QG.compute_local_descriptors(method='all_qg_cluster', data='coords', neighborhood='pointcloudgraph', scale=10)

In [ ]:
k = [k for k in QG.nodes[list(QG.nodes)[0]] if 'planar' in k or 'linear' in k or 'scatter' in k]
print('descriptor keys:', k)

## 5. Exports (descriptors)

Export the descriptors on the quotient graph and a matplotlib rendering.


In [ ]:
display_and_export_quotient_graph_matplotlib(qg=QG, node_sizes=20, name="planarity",
                                             data_on_nodes='planarity', data=True, attributekmeans4clusters=False, directory=OUTPUT_DIR)
export_quotient_graph_attribute_on_point_cloud(QG, 'planarity', directory=OUTPUT_DIR)
export_quotient_graph_attribute_on_point_cloud(QG, 'linearity', directory=OUTPUT_DIR)
export_quotient_graph_attribute_on_point_cloud(QG, 'scattering', directory=OUTPUT_DIR)


## 6. Build the minimum spanning tree of the quotient graph

The semantic labels are inferred along a tree. We compute the MST over the QG (with semantic weights)
and import it into the tree representation used by the HMM/Viterbi code.


In [ ]:
QG_t2 = minimum_spanning_tree_quotientgraph_semantics(QG)
st_tree = V.read_pointcloudgraph_into_treex(pointcloudgraph=QG_t2, directory=OUTPUT_DIR)

In [ ]:
rt = 8
t = V.build_spanning_tree(st_tree, rt, list_att=['planarity', 'linearity', 'scattering'])
V.create_observation_list(t, list_obs=['planarity', 'linearity', 'scattering'])
print('spanning tree built, observations created')

## 7. HMM parameters & Viterbi decoding

Gaussian emission probabilities per semantic state, a 3-state transition matrix
(leaf / stem / NSP), and the Viterbi algorithm over the tree.


In [ ]:
initial_distribution = [1, 0, 0]
transition_matrix = [[0.2, 0, 0.8],
                     [0, 0.8, 0.2],
                     [0, 0.8, 0.2]]

In [ ]:
def pdf_emission_dim1(x, moy, sd):
    return 1.0 / (sd * math.sqrt(2 * math.pi)) * math.exp(-1.0 / (2 * sd ** 2) * (x - moy) ** 2)

In [ ]:
def pdf_emission_dimn(x, k, parameterstot):
    if len(parameterstot) == 1:
        return pdf_emission_dim1(x, moy=parameterstot[0][k][0], sd=parameterstot[0][k][1])
    p = 1
    for i in range(len(parameterstot[0])):
        p *= pdf_emission_dim1(x[i], moy=parameterstot[k][i][0], sd=parameterstot[k][i][1])
    return p

In [ ]:
parameterstot = [[[0.1, 0.9], [0.8, 0.9], [0.01, 0.3]],
                 [[0.8, 0.9], [0.5, 0.9], [0.01, 0.3]],
                 [[0.1, 0.9], [0.9, 0.9], [0.1, 0.3]]]

In [ ]:
V.viterbi(t, 'observations', initial_distribution, transition_matrix, pdf_emission_dimn, parameterstot)
V.add_viterbi_results_to_quotient_graph(QG_t2, t, list_semantics=['leaf', 'stem', 'NSP'])
V.add_viterbi_results_to_quotient_graph(QG, t, list_semantics=['leaf', 'stem', 'NSP'])
print('Viterbi decoding done')
print('viterbi_class per QG node:', Counter(int(QG.nodes[n]['viterbi_class']) for n in QG.nodes))

## 8. Display results

Render the quotient graph coloured by Viterbi semantic class and export the per-point labels.


In [ ]:
display_and_export_quotient_graph_matplotlib(qg=QG, node_sizes=20, name="quotient_graph_viterbi",
                                             data_on_nodes='viterbi_class', data=True, attributekmeans4clusters=False, directory=OUTPUT_DIR)
export_quotient_graph_attribute_on_point_cloud(QG, attribute='viterbi_class', directory=OUTPUT_DIR)
print('total elapsed: %.1fs' % (time.time() - begin))